
# 🔬 Proyecto K-Means: Del Playground Web al Análisis con Python

Este cuaderno de Jupyter actúa como el puente analítico para nuestro experimento de **aprendizaje no supervisado**. Su objetivo es cargar, visualizar y evaluar de forma matemática los datos geométricos interactivos que diseñamos y exportamos desde nuestra aplicación web **K-Means Playground**.

A través de este flujo, compararemos la clasificación intuitiva realizada en la interfaz gráfica contra el entrenamiento formal utilizando la biblioteca `scikit-learn`

---

In [ ]:
%pip install pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
# Configuración visual para que los gráficos se vean modernos
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Cargar el dataset generado por ti en la web
df = pd.read_csv("datos_dibujados_kmeans.csv")

In [ ]:
# Mostrar las primeras filas para verificar que se exportó bien
print("Dimensiones del dataset:", df.shape)
df.head()

**n_clusters=k_optimo**
¿Qué hace?: Define el número de grupos (clusters) en los que vamos a dividir los datos.
En la práctica: Le indicamos cuántos "colores" o categorías diferentes va a crear. Utiliza el valor almacenado en la variable k_optimo que vimos tras analizar los datos.

In [ ]:
k_optimo = 4

## 🧠 Marco Teórico del Experimento (Paso a Paso)

El algoritmo **K-medias (K-means)** es uno de los métodos más populares de **agrupamiento basado en prototipos**. A continuación, se detalla el proceso matemático y metodológico que ejecuta el algoritmo detrás de escena:

### **Paso 1: Inicialización de los Centroides**
El algoritmo comienza seleccionando un número de grupos ($K$) predefinido. Se colocan $K$ puntos en el espacio geométrico que actuarán como centros iniciales o "prototipos". 
* *Nota metodológica:* K-medias es altamente **sensible a las posiciones iniciales** de estos centroides. Una mala inicialización puede hacer que el algoritmo converja en un mínimo local subóptimo en lugar del óptimo global.


### **Paso 2: Fase de Asignación (Métrica de Distancia)**
Cada punto de datos disponible en nuestro dataset se asigna al centroide que tenga más cerca. Para medir la cercanía en un espacio continuo, calculamos la **Distancia Euclídea** entre un punto $x$ y un centroide $\mu_j$:

$$d(x, \mu_j) = \sqrt{\sum_{i=1}^{n} (x_i - \mu_{ji})^2}$$

**init='k-means++'**
¿Qué hace?: Selecciona de forma inteligente las posiciones de inicio para los primeros centroides.
En lugar de colocar los puntos de partida de los grupos de manera completamente al azar, los coloca lo más alejados posible unos de otros. Esto hace que el algoritmo trabaje mucho más rápido y evita que los grupos queden mal distribuidos desde el principio.

**random_state=42**
¿Qué hace?: Establece una semilla fija para el generador de números aleatorios.
Asegura la reproducibilidad. Como el algoritmo tiene pasos iniciales que dependen del azar, si no fijas este valor, podrías obtener resultados ligeramente distintos cada vez que ejecutes el código. Al usar 42 (o cualquier otro número), garantizas que el código siempre clasifique los datos exactamente de la misma manera.

**n_init=10**
¿Qué hace?: Determina cuántas veces consecutivas se ejecutará todo el algoritmo con diferentes puntos de partida.
Python ejecutará el proceso de K-medias 10 veces en segundo plano. Al terminar, comparará los 10 intentos, descartará los 9 peores y se quedará únicamente con la ejecución que haya logrado agrupar los datos con la menor dispersión o error posible. Esto asegura que la solución final sea altamente confiable.

In [ ]:
kmeans = KMeans(n_clusters=k_optimo, init='k-means++', random_state=42, n_init=10)


**df[['X', 'Y']] (La selección de coordenadas)**
Qué hace: Extrae únicamente las columnas de coordenadas X e Y de tu DataFrame df (que contiene los puntos que exportaste desde tu Playground web).
El doble corchete [['X', 'Y']] es fundamental en Pandas. Asegura que el resultado sea una estructura de dos dimensiones (una matriz de dos columnas), que es el formato exacto que exige Scikit-Learn para poder procesar las distancias geométricas.

**.fit_predict(...) (Entrenamiento y asignación "2 en 1")**
Este es un método optimizado de Scikit-Learn que combina dos acciones en una sola llamada de manera muy eficiente:
fit (Entrenar): El algoritmo de K-medias toma tus coordenadas X e Y y ejecuta su proceso iterativo. Coloca los centroides en sus posiciones iniciales, calcula la distancia de cada punto a ellos, mueve los centroides al promedio geométrico de sus grupos y repite este ciclo hasta que alcanzan la convergencia (cuando ya no cambian de posición).
**predict (Asignar):** Una vez que los grupos están perfectamente definidos y los centroides están en su lugar definitivo, el algoritmo le asigna a cada punto un número de grupo (por ejemplo, 0, 1 o 2 si definiste tres grupos).

**df['Cluster_Python'] = ... (Creación de la columna de resultados)**
Qué hace: Crea una columna completamente nueva en tu tabla de datos llamada Cluster_Python y guarda en ella la lista de etiquetas numéricas (0, 1, 2...) que devolvió el paso anterior para cada fila.
-> Esto es la clave de tu análisis. Nos permite tener en un mismo DataFrame las coordenadas del punto, el grupo que le asignamos visualmente en la web (Cluster) y el grupo que calculó de manera estricta el algoritmo en Python (Cluster_Python), lo que hace facilísimo comparar ambos resultados.

In [ ]:
df['Cluster_Python'] = kmeans.fit_predict(df[['X', 'Y']])
# Comparar cómo clasificó la Web vs cómo clasificó Python
df.head()

In [ ]:
# Mostrar las iteraciones reales hasta que el modelo convergió
print('Iterations until convergence:', kmeans.n_iter_)

# Mostrar las coordenadas finales de los centroides calculados por Python
print('Final centroids:\n', kmeans.cluster_centers_)

## 💡 ¿Cómo interpretar la salida de los centroides?
Si tu modelo se entrenó con un K=3 en un espacio bidimensional (usando tus columnas X e Y), al imprimir kmeans.cluster_centers_ verás una matriz como esta:

* **Final centroids:**
 [[150.23  80.45]
  [512.11 390.62]
  [320.50 210.15]]

## 📉 La Función Objetivo: Minimización de la Dispersión (SSE)

El objetivo matemático de K-medias es encontrar una partición de los datos que minimice la **Suma de Errores al Cuadrado (SSE)**, también conocida como **Inercia** o dispersión dentro de los grupos:

$$SSE = \sum_{j=1}^{K} \sum_{x_i \in S_j} \|x_i - \mu_j\|^2$$

* **En nuestra Web:** La aplicación calcula el SSE en tiempo real tras cada clic. Verás que este valor disminuye drásticamente a medida que los centroides se acomodan en el centro de gravedad de sus nubes de datos.
* **En este Notebook:** Utilizaremos la inercia calculada por `scikit-learn` para validar el comportamiento del modelo.

---
## 🛠️ Buenas Prácticas Aplicadas: Escalado RobustScaler

En problemas reales (como la segmentación de datos financieros o de clientes), las variables suelen tener magnitudes y escalas muy diferentes. Dado que K-medias se basa en distancias métricas, las variables con números más grandes dominarán la ecuación si no se realiza un preprocesamiento adecuado.

* **El Problema de los Outliers:** El uso de escaladores tradicionales como `StandardScaler` se ve severamente afectado por valores extremos (outliers).
* **La Solución:** Implementamos **`RobustScaler`** . Este escalador utiliza estadísticos robustos como la **mediana** y el **rango intercuartílico (IQR)** para normalizar las características de manera segura frente a anomalías:

$$\hat{x} = \frac{x - \text{mediana}(x)}{IQR}$$

Esto garantiza que las distancias calculadas por nuestro modelo sean equilibradas y representen la verdadera estructura de los datos.

---

In [ ]:
# Crear una gráfica con dos paneles para comparar la clasificación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

In [ ]:
# Crear una gráfica con dos paneles para comparar la clasificación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Clasificación de la Web
sns.scatterplot(
    data=df, 
    x='X', 
    y='Y', 
    hue='Cluster', 
    palette='viridis', 
    ax=axes[0]
)
axes[0].set_title('Clasificación Web')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# Panel 2: Clasificación de Python (Scikit-Learn)
sns.scatterplot(
    data=df, 
    x='X', 
    y='Y', 
    hue='Cluster_Python', 
    palette='viridis', 
    ax=axes[1]
)
axes[1].set_title('Clasificación Python (K-Means)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')

plt.tight_layout()
plt.show()

<div align="center">
  <img src="img/Captura%20de%20pantalla%202026-08-15%20095006.jpg" width="100%">
</div>

## 🎯 Evaluación de la Calidad del Agrupamiento

Para determinar de forma científica si el número de grupos ($K$) seleccionado es el adecuado sin sesgos de intuición, aplicaremos dos técnicas heurísticas clave:

1. **El método del Codo (Elbow Method):** Evaluamos el SSE para diferentes valores de $K$. Buscaremos el "punto de inflexión" donde la reducción del error empieza a estabilizarse de manera significativa.
2. **Análisis de Silueta (Silhouette):** Mide la cohesión interna del grupo frente a su separación con los grupos vecinos. Un coeficiente cercano a $1$ indica una asignación óptima de los clusters.
